# Smart Study Advisor — Exploratory Data Analysis

**Project:** Study Strategy Recommendation (Machine Learning)  
**Dataset:** `datasets/student_study_strategy.csv`  
**Target:** `RecommendedStrategy` → Rest | BalancedStudy | IntensiveStudy | LongTermPlan

---

This notebook performs a structured EDA before preprocessing and model training.
All figures are exported to `ml/output/figures/` and a summary report to `ml/output/eda_summary.md`.

> **Run from the `ml/` directory** (or ensure `ML_ROOT` resolves correctly below).

In [ ]:
from pathlib import Path
import sys

ML_ROOT = Path.cwd()
if ML_ROOT.name == "notebooks":
    ML_ROOT = ML_ROOT.parent
sys.path.insert(0, str(ML_ROOT))

import pandas as pd
from IPython.display import Image, display, Markdown

from src.analysis.eda_helpers import (
    STRATEGY_ORDER,
    build_eda_summary,
    class_balance_table,
    column_overview_table,
    configure_plot_style,
    correlation_highlights,
    descriptive_statistics,
    detect_iqr_outliers,
    duplicate_summary,
    encode_for_correlation,
    ensure_output_dirs,
    invalid_value_report,
    missing_value_summary,
    outlier_rows,
)
from src.analysis.eda_plots import (
    plot_categorical_count,
    plot_categorical_vs_strategy,
    plot_encoded_correlation_matrix,
    plot_feature_vs_strategy,
    plot_missing_values,
    plot_numeric_boxplot_by_strategy,
    plot_numeric_histogram,
    plot_outlier_summary,
    plot_target_distribution,
)
from src.data.schema import (
    CATEGORICAL_FEATURES,
    FEATURE_COLUMNS,
    NUMERIC_FEATURES,
    TARGET_COLUMN,
    load_dataset,
    load_schema,
)

configure_plot_style()
FIGURES_DIR, OUTPUT_DIR = ensure_output_dirs(ML_ROOT)
schema = load_schema()
df = load_dataset()

def show(path: Path, width: int = 900):
    display(Image(filename=str(path), width=width))

print(f"Dataset: {schema.source_path}")
print(f"Figures: {FIGURES_DIR}")

---
## 1. Dataset overview

Inspect shape, sample records, column meanings, and data types.

In [ ]:
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
display(column_overview_table(FEATURE_COLUMNS, TARGET_COLUMN))
display(df.dtypes.to_frame("DataType"))
df.head(10)

**Observations:** The dataset contains 400 labelled records with 8 input features and one categorical target. Four features are numeric (`HoursStudied`, `NumberOfSubjects`, `DaysUntilExam`, `SleepHours`) and four are categorical (`StressLevel`, `FatigueLevel`, `SleepQuality`, `PreviousFeedback`). Sample rows show coherent combinations — e.g. high stress/fatigue with poor sleep mapping to `Rest`, or many days until exam mapping to `LongTermPlan`.

---
## 2. Data quality checks

Verify completeness, uniqueness, valid categories, and class balance.

In [ ]:
missing_df = missing_value_summary(df)
missing_display = missing_df.copy()
missing_display["MissingPercent"] = missing_display["MissingPercent"].astype(str) + "%"
display(missing_display)

dup = duplicate_summary(df)
print(f"Duplicate rows: {dup['duplicate_rows']}")
print(f"Duplicate feature rows (excl. target): {dup['duplicate_feature_rows']}")

display(invalid_value_report(df))
display(class_balance_table(df, TARGET_COLUMN))

missing_fig = plot_missing_values(df, FIGURES_DIR)
show(missing_fig)

**Observations:** Data quality is excellent — **0 missing values**, **0 duplicate rows**, and **0 invalid categorical values**. All four target classes contain exactly 100 records (25% each), ensuring balanced evaluation metrics. The flat missing-value chart confirms a complete dataset; no imputation is required.

---
## 3. Descriptive statistics

Mean, median, standard deviation, min, max, and quartiles for numeric features.

In [ ]:
desc_stats = descriptive_statistics(df, NUMERIC_FEATURES)
display(desc_stats)

**Observations:**
- `DaysUntilExam` spans 0–30 days (mean ≈ 8.5) with high variance — captures both urgent and long-term scenarios.
- `HoursStudied` averages ≈ 4.3 h/day with a wide range (0–10).
- `SleepHours` clusters around 6.4 h (IQR ≈ 5.5–7.5).
- `NumberOfSubjects` averages ≈ 4.6 (1–9 subjects).

---
## 4. Target distribution

In [ ]:
balance = class_balance_table(df, TARGET_COLUMN)
display(balance)
target_fig = plot_target_distribution(df, TARGET_COLUMN, FIGURES_DIR)
show(target_fig)

**Observations:** Perfect class balance — each strategy has exactly 100 samples (25%). Accuracy and macro-F1 are appropriate primary metrics; no resampling is needed.

---
## 5. Numerical feature analysis

For each numeric feature: histogram (distribution) and boxplot (by strategy).

In [ ]:
for col in NUMERIC_FEATURES:
    print(f"\n{'='*60}\n{col}\n{'='*60}")
    show(plot_numeric_histogram(df, col, FIGURES_DIR), 800)
    show(plot_numeric_boxplot_by_strategy(df, col, TARGET_COLUMN, FIGURES_DIR), 800)

**Observations (numerical features):**

| Feature | Interpretation |
|---------|----------------|
| **HoursStudied** | Bimodal pattern — low values for `Rest`/`LongTermPlan`, high for `IntensiveStudy`. Strong discriminator. |
| **NumberOfSubjects** | `IntensiveStudy` shows the highest median (5–8 subjects); `Rest` the lowest. |
| **DaysUntilExam** | Clearest separator — `LongTermPlan` at 14–30 days; `Rest`/`IntensiveStudy` below 5 days. |
| **SleepHours** | `Rest` median ≈ 4–5 h; `LongTermPlan` ≈ 7.5–8 h. Overlaps exist in mid-range for `BalancedStudy`. |

---
## 6. Categorical feature analysis

Count plots and cross-tabulations with `RecommendedStrategy`.

In [ ]:
for col in CATEGORICAL_FEATURES:
    print(f"\n{'='*60}\n{col}\n{'='*60}")
    display(df[col].value_counts().to_frame("Count"))
    show(plot_categorical_count(df, col, FIGURES_DIR), 700)
    show(plot_categorical_vs_strategy(df, col, TARGET_COLUMN, FIGURES_DIR), 850)

**Observations (categorical features):**

| Feature | Interpretation |
|---------|----------------|
| **StressLevel** | `High` dominates `Rest` and `IntensiveStudy`; `Low` dominates `LongTermPlan`. |
| **FatigueLevel** | `High` almost exclusively maps to `Rest`; `Low` to `LongTermPlan`. |
| **SleepQuality** | `Poor` concentrated in `Rest`; `Good` in `LongTermPlan` and `BalancedStudy`. |
| **PreviousFeedback** | `Negative` appears mostly in `Rest` and `BalancedStudy` (safer shift after bad feedback). `Positive` common in `IntensiveStudy`. |

---
## 7. Correlation analysis

Categorical features are **ordinal-encoded for correlation only** (not for modelling):
- Stress/Fatigue: Low=0, Medium=1, High=2
- SleepQuality: Poor=0, Average=1, Good=2
- PreviousFeedback: None=0, Positive=1, Mixed=2, Negative=3

In [ ]:
encoded = encode_for_correlation(df, FEATURE_COLUMNS)
corr = encoded[FEATURE_COLUMNS].corr().round(3)
display(corr)

pos, neg = correlation_highlights(corr)
print("Strongest POSITIVE correlations:")
for a, b, v in pos[:5]:
    print(f"  {a} ↔ {b}: {v:.3f}")
print("\nStrongest NEGATIVE correlations:")
for a, b, v in neg[:5]:
    print(f"  {a} ↔ {b}: {v:.3f}")

show(plot_encoded_correlation_matrix(encoded, FEATURE_COLUMNS, FIGURES_DIR), 950)

**Observations:**
- **Positive:** `SleepHours` ↔ `SleepQuality` — longer sleep aligns with better quality ratings.
- **Positive:** `NumberOfSubjects` ↔ `HoursStudied` — heavier workloads correlate with more study time.
- **Negative:** `DaysUntilExam` ↔ `StressLevel` — urgency increases perceived stress.
- **Negative:** `DaysUntilExam` ↔ `FatigueLevel` — more time before exams associates with lower fatigue.

These relationships match domain expectations and confirm the dataset's internal consistency.

---
## 8. Outlier analysis (IQR method)

Outlier if value < Q1 − 1.5×IQR or > Q3 + 1.5×IQR.

In [ ]:
outlier_summary = detect_iqr_outliers(df, NUMERIC_FEATURES)
display(outlier_summary)

outlier_sample = outlier_rows(df, NUMERIC_FEATURES)
print(f"Rows flagged: {len(outlier_sample)} ({len(outlier_sample)/len(df)*100:.1f}%)")
if len(outlier_sample) > 0:
    display(outlier_sample.head(10))

show(plot_outlier_summary(outlier_summary, FIGURES_DIR))

**Observations & recommendation:** Outliers represent valid extreme student states (e.g. 10 h study day, exam today). **Retain them** — removing would discard edge cases the advisor must handle. If counts were high due to recording errors, winsorisation could be considered; that is not the case here.

---
## 9. Feature relationship plots

Focused views of the most decision-relevant features against `RecommendedStrategy`.

In [ ]:
relationships = [
    ("SleepQuality", "09_sleep_quality_vs_strategy.png", False),
    ("StressLevel", "10_stress_level_vs_strategy.png", False),
    ("FatigueLevel", "11_fatigue_level_vs_strategy.png", False),
    ("DaysUntilExam", "12_days_until_exam_vs_strategy.png", True),
    ("SleepHours", "13_sleep_hours_vs_strategy.png", True),
]
for feature, fname, is_numeric in relationships:
    show(plot_feature_vs_strategy(df, feature, TARGET_COLUMN, FIGURES_DIR, fname, numeric=is_numeric), 850)

**Observations:** These five plots confirm the primary decision boundaries:
1. Poor sleep → `Rest`; Good sleep → `LongTermPlan` / `BalancedStudy`.
2. High stress → `Rest` or `IntensiveStudy` (depending on fatigue).
3. High fatigue → almost exclusively `Rest`.
4. Few days until exam → `Rest` / `IntensiveStudy`; many days → `LongTermPlan`.
5. Low sleep hours → `Rest`; adequate sleep → other strategies.

---
## 10. Export summary report

In [ ]:
summary_text = build_eda_summary(
    df=df,
    schema_path=schema.source_path,
    figures_dir=FIGURES_DIR,
    numeric_columns=NUMERIC_FEATURES,
    categorical_columns=CATEGORICAL_FEATURES,
    target_column=TARGET_COLUMN,
    missing_df=missing_value_summary(df),
    duplicate_info=dup,
    invalid_df=invalid_value_report(df),
    balance_df=balance,
    desc_stats=desc_stats,
    outlier_summary=outlier_summary,
    outlier_sample=outlier_sample,
    encoded_corr=encoded[FEATURE_COLUMNS].corr(),
)

report_path = OUTPUT_DIR / "eda_summary.md"
report_path.write_text(summary_text, encoding="utf-8")

figures = sorted(p.name for p in FIGURES_DIR.glob("*.png"))
print(f"Report saved: {report_path}")
print(f"Figures exported: {len(figures)}")
for f in figures:
    print(f"  - {f}")

---
## Conclusion

| Aspect | Result |
|--------|--------|
| Data quality | Complete, valid, no duplicates |
| Class balance | 100 per strategy (25% each) |
| Key discriminators | DaysUntilExam, StressLevel, FatigueLevel, SleepQuality |
| Outliers | Retain — valid extreme cases |
| Next step | **Step 4:** preprocessing + baseline classifier |

Alternatively, regenerate all outputs in one command:
```bash
python scripts/run_eda.py
```